# 淘到宝引擎评委短邀请码生成器

短邀请码采用 `tdb-xxxxx` 格式，必须登记到云端数据库后才能使用。本 Notebook 通过 SSH 在服务器内创建，不会把数据库密码或签名密钥复制到本机。

In [ ]:
from pathlib import Path
import subprocess

# 只需要调整这里。
VALID_DAYS = 20
MAX_USES = 100
COUNT = 1
SSH_TARGET = 'root@8.218.59.190'
SSH_KEY = Path.home() / '.ssh' / 'taodaobao-deploy.pem'

In [ ]:
if not 1 <= VALID_DAYS <= 30:
    raise ValueError('VALID_DAYS 必须在 1 至 30 之间')
if not 1 <= MAX_USES <= 100:
    raise ValueError('MAX_USES 必须在 1 至 100 之间')
if not 1 <= COUNT <= 100:
    raise ValueError('COUNT 必须在 1 至 100 之间')
if not SSH_KEY.is_file():
    raise FileNotFoundError(f'找不到 SSH 密钥：{SSH_KEY}')

remote_command = (
    "set -eu; "
    "image=$(docker inspect taodaobao-app --format '{{.Config.Image}}'); "
    "network=$(docker inspect taodaobao-app --format '{{range $k, $v := .NetworkSettings.Networks}}{{$k}}{{end}}'); "
    "docker run --rm --network \"$network\" "
    "--env-file /opt/taodaobao/runtime.env --entrypoint python \"$image\" "
    f"-m app.cli.create_short_invitation --days {VALID_DAYS} "
    f"--max-uses {MAX_USES} --count {COUNT}"
)
result = subprocess.run(
    ['ssh', '-i', str(SSH_KEY), '-o', 'StrictHostKeyChecking=accept-new', SSH_TARGET, remote_command],
    check=True, capture_output=True, text=True, encoding='utf-8',
)
codes = [line.strip() for line in result.stdout.splitlines() if line.startswith('tdb-')]
if len(codes) != COUNT:
    raise RuntimeError(f'服务端未返回预期数量的短邀请码：{result.stderr}')
print(f'已登记 {COUNT} 个邀请码：有效 {VALID_DAYS} 天，每个最多使用 {MAX_USES} 次')
for index, code in enumerate(codes, 1):
    print(f'{index}. {code}')